In [ ]:
# Prepare to load data from google drive
from google.colab import drive

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
current_step = 'step_002'
workDir = f'{gdrive_path}/My Drive/Research/{current_step}'

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
import multiprocessing
import os

print(multiprocessing.cpu_count())
print(os.cpu_count())
print(len(os.sched_getaffinity(0)))

In [ ]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

In [ ]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


In [ ]:
!pip install -e {model_path}

In [ ]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


In [ ]:
!pip install -e {trainner_path}

In [ ]:
path = os.path.join(trainner_path, "logs")
if os.path.isdir(path):
  shutil.rmtree(path)

path = os.path.join(trainner_path, "research_logs/DarwinOp3-v1")
if os.path.isdir(path):
  shutil.rmtree(path)


In [ ]:
# Hyper Parameters Tunning
%cd {trainner_path}
# !python3 train.py --algo ppo --env DarwinOp3-v1 --conf-file research/config/ppo.yml --device cpu -n 100000 -optimize --n-trials 500 --n-jobs 4 --sampler tpe --pruner median
# !python3 train.py --algo ppo --env DarwinOp3-v0 --conf-file research/config/ppo.yml --device cpu -optimize -n 100000 -optimize --n-trials 500 --n-jobs 4 --study-name Darwin --storage sqlite:///tunning.db
# !./train_ppo.sh
# !./train_a2c.sh

algo = 'a2c'
config = f'research_config/{algo}.yml'

!python3 train.py --algo $algo --env DarwinOp3-v2 -conf $config --tensorboard-log research_logs/ --device cpu --save-freq 100000 --vec-env subproc --eval-freq 200000 --n-eval-envs 1 --eval-episodes 20 --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 forward_velocity_weight:5.0 ctrl_cost_weight:1e-3
# !python3 train.py --log-interval 1000 --algo $algo --env DarwinOp3-v1 -conf $config --tensorboard-log research_logs/ --save-freq 100000 --vec-env subproc --eval-freq 200000 --n-eval-envs 1 --eval-episodes 20 --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 forward_velocity_weight:5.0 ctrl_cost_weight:1e-3

from google.colab import files
%cd /content/

# DEFINE WORK DIRECTORY
rlogs = f'{algo}_rlogs.zip'
logs = f'{algo}_logs.zip'

!ls
!zip -r $rlogs rl-zoo/research_logs/DarwinOp3-v1
shutil.copy2(f'/content/{rlogs}', workDir)

!zip -r $logs rl-zoo/logs
shutil.copy2(f'/content/{logs}', workDir)
